# Lab 10: MEPS Uninsurance Rate Analysis

## AI and Machine Learning for Quantitative Macroeconomics

**Goal:** Use Claude Code to analyze US uninsurance rates from 2000–2023 using MEPS data.

**Duration:** 1 hour

**What you will learn:**
- How to direct an AI agent through a multi-step data analysis workflow
- How to validate AI-generated output against known benchmarks
- The division of labor: you provide the question and domain knowledge; the agent handles the mechanics

---

### Background

The Medical Expenditure Panel Survey (MEPS), conducted by the Agency for Healthcare Research and Quality (AHRQ), is the most complete source of data on healthcare utilization, expenditure, and insurance coverage in the United States. The Full Year Consolidated files contain person-level records with insurance coverage status, demographics, and survey weights for nationally representative estimates.

**Research Question:** Who is uninsured in the US, 2000–2023? How has the uninsurance rate evolved, and how did the Affordable Care Act change the distribution?

**Connection to PaulGP's HMDA article:** This lab follows the same workflow arc — Problem → Download → Harmonize → Analyze → Visualize — but applied to health insurance data instead of mortgage data.

---

### Prerequisites

- Claude Code installed (Pro or Max subscription)
- Python 3.8+ with `pandas`, `matplotlib`, `numpy`
- `pyreadstat` for reading SAS transport files (`pip install pyreadstat`)
- Internet access for downloading MEPS files from AHRQ

## Step 0: Setup

First, install required packages and set up the project.

In [ ]:
# Install pyreadstat if not already installed
# !pip install pyreadstat

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## Step 1: Data Acquisition (Prompt 1)

### The Claude Code Prompt

In a Claude Code session, you would give this prompt:

```
I want to analyze uninsurance rates in the US from 2000 to 2023
using MEPS data from AHRQ.

Download the Full Year Consolidated files for each year.
The files are in SAS transport format (.ssp).

Parse each file. Extract these variables:
- Insurance coverage (INSCOV / INSCOVY)
- Age (AGELAST)
- Race/ethnicity (RACETHX)
- Poverty category (POVCAT)
- Census region (REGION)
- Employment status (EMPST)
- Sex (SEX)
- Person-level weight (PERWT / PERWTF)

Harmonize variable names across years (they are year-suffixed,
e.g., INSCOV01, INSCOV22).

Save as a single parquet file: meps_panel_2000_2023.parquet
```

### What Claude Does

Claude Code would:
1. Discover the AHRQ download URL pattern
2. Download 24 SAS transport files (one per year)
3. Build a variable crosswalk dictionary for year-specific names
4. Parse, harmonize, and merge into a single DataFrame

### For This Lab

Below we show the code Claude would generate. In the live lab, you let Claude write this code — here we provide it for reference and reproducibility.

In [ ]:
# ============================================================
# MEPS Variable Crosswalk
# ============================================================
# MEPS variable names are year-suffixed. This crosswalk maps
# year-specific names to standardized names.
#
# Key variable: INSCOV (Insurance Coverage)
#   1 = Any Private insurance
#   2 = Public Only
#   3 = Uninsured
#
# This is the kind of tedious, error-prone work that Claude
# Code handles well — and where silent errors are most dangerous.
# ============================================================

def build_crosswalk(year):
    """Build variable name crosswalk for a given MEPS year."""
    yy = str(year)[-2:]  # e.g., '01' for 2001, '22' for 2022
    
    crosswalk = {
        # Insurance coverage
        f'INSCOV{yy}': 'INSCOV',
        f'INSCOVY{yy[-1]}': 'INSCOV',  # alternate naming
        'INSCOV': 'INSCOV',
        
        # Person weight
        f'PERWT{yy}F': 'PERWT',
        'PERWTF': 'PERWT',
        
        # Age
        'AGELAST': 'AGE',
        f'AGE{yy}X': 'AGE',
        'AGE31X': 'AGE',
        'AGE42X': 'AGE',
        'AGE53X': 'AGE',
        
        # Poverty category
        f'POVCAT{yy}': 'POVCAT',
        'POVCAT': 'POVCAT',
        
        # Race/ethnicity
        'RACETHX': 'RACETHX',
        'RACEX': 'RACETHX',
        f'RACETHN{yy}': 'RACETHX',
        
        # Region
        f'REGION{yy}': 'REGION',
        'REGION': 'REGION',
        
        # Employment
        'EMPST31': 'EMPST',
        'EMPST42': 'EMPST',
        'EMPST53': 'EMPST',
        f'EMPST{yy}': 'EMPST',
        
        # Sex
        'SEX': 'SEX',
    }
    
    return crosswalk

print('Crosswalk function defined.')
print(f'Example for 2022: INSCOV22 -> {build_crosswalk(2022).get("INSCOV22", "not found")}')
print(f'Example for 2001: PERWT01F -> {build_crosswalk(2001).get("PERWT01F", "not found")}')

### Data Download Function

MEPS Full Year Consolidated files are available from AHRQ as SAS transport (.ssp) files inside ZIP archives. The file numbering (HC-NNN) maps to survey years.

**Note:** In the live lab, Claude Code discovers this URL pattern automatically. Here we provide the mapping explicitly.

In [ ]:
import urllib.request
import zipfile
import io
import pyreadstat

# MEPS HC file number mapping (Full Year Consolidated)
# Source: https://meps.ahrq.gov/mepsweb/data_stats/download_data_files.jsp
HC_FILE_MAP = {
    2000: 'h50',   2001: 'h60',   2002: 'h70',   2003: 'h79',
    2004: 'h89',   2005: 'h97',   2006: 'h105',  2007: 'h113',
    2008: 'h121',  2009: 'h129',  2010: 'h138',  2011: 'h147',
    2012: 'h155',  2013: 'h163',  2014: 'h171',  2015: 'h181',
    2016: 'h192',  2017: 'h201',  2018: 'h209',  2019: 'h216',
    2020: 'h224',  2021: 'h233',  2022: 'h243',  2023: 'h252',
}

def download_meps_year(year, data_dir='meps_data'):
    """Download and parse one year of MEPS Full Year Consolidated data."""
    Path(data_dir).mkdir(exist_ok=True)
    
    hc_file = HC_FILE_MAP.get(year)
    if hc_file is None:
        raise ValueError(f'No HC file mapping for year {year}')
    
    url = f'https://meps.ahrq.gov/mepsweb/data_files/pufs/{hc_file}ssp.zip'
    local_path = Path(data_dir) / f'{hc_file}ssp.zip'
    
    # Download if not cached
    if not local_path.exists():
        print(f'  Downloading {url} ...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = response.read()
        local_path.write_bytes(data)
    
    # Extract and read SAS transport file
    with zipfile.ZipFile(local_path) as zf:
        ssp_names = [n for n in zf.namelist() if n.lower().endswith('.ssp')]
        if not ssp_names:
            raise FileNotFoundError(f'No .ssp file in {local_path}')
        with zf.open(ssp_names[0]) as ssp_file:
            df, meta = pyreadstat.read_xport(io.BytesIO(ssp_file.read()))
    
    return df

print(f'Download function defined. {len(HC_FILE_MAP)} years mapped (2000-2023).')

In [ ]:
# ============================================================
# Extract and harmonize variables for one year
# ============================================================

STANDARD_COLS = ['INSCOV', 'AGE', 'RACETHX', 'POVCAT', 'REGION', 'EMPST', 'SEX', 'PERWT']

def extract_and_harmonize(df, year):
    """Extract target variables using the year-specific crosswalk."""
    crosswalk = build_crosswalk(year)
    
    # Uppercase all column names for matching
    df.columns = [c.upper() for c in df.columns]
    
    result = {}
    for col_name in df.columns:
        if col_name in crosswalk:
            std_name = crosswalk[col_name]
            if std_name not in result:  # first match wins
                result[std_name] = df[col_name]
    
    out = pd.DataFrame(result)
    out['YEAR'] = year
    
    # Keep only rows with valid insurance coverage (1, 2, or 3)
    if 'INSCOV' in out.columns:
        out = out[out['INSCOV'].isin([1, 2, 3])]
    
    return out

print('Extraction function defined.')

In [ ]:
# ============================================================
# Download and merge all years
# WARNING: This downloads ~2-4 GB of data. Takes 10-30 minutes
# depending on your connection.
#
# For the lab, you may start with a subset of years:
#   years = [2000, 2005, 2010, 2013, 2014, 2015, 2019, 2022]
# ============================================================

# Full range
years = list(range(2000, 2024))

# Or use a subset for faster lab execution:
# years = [2000, 2005, 2010, 2013, 2014, 2015, 2019, 2022]

all_dfs = []
for year in years:
    try:
        print(f'Processing {year}...', end=' ')
        raw = download_meps_year(year)
        harmonized = extract_and_harmonize(raw, year)
        all_dfs.append(harmonized)
        print(f'{len(harmonized):,} records, '
              f'cols: {sorted(harmonized.columns.tolist())}')
    except Exception as e:
        print(f'ERROR: {e}')

panel = pd.concat(all_dfs, ignore_index=True)
print(f'\nMerged panel: {len(panel):,} rows, {len(panel.columns)} columns')
print(f'Years: {sorted(panel["YEAR"].unique())}')
print(f'Columns: {sorted(panel.columns.tolist())}')

In [ ]:
# Save to parquet for reuse
panel.to_parquet('meps_panel_2000_2023.parquet', index=False)
print(f'Saved: meps_panel_2000_2023.parquet ({Path("meps_panel_2000_2023.parquet").stat().st_size / 1e6:.1f} MB)')

### Checkpoint (15 min)

At this point you should have:
- A parquet file with 500K+ rows
- Columns: YEAR, INSCOV, AGE, RACETHX, POVCAT, REGION, EMPST, SEX, PERWT
- INSCOV values in {1, 2, 3}

**Quick validation:**

In [ ]:
# Quick validation
print('Insurance coverage distribution (unweighted counts):')
print(panel['INSCOV'].value_counts().sort_index())
print(f'\nINSCOV values: {sorted(panel["INSCOV"].unique())}')
print(f'Year range: {panel["YEAR"].min()} - {panel["YEAR"].max()}')
print(f'Total records: {len(panel):,}')

## Step 2: Analysis (Prompt 2)

### The Claude Code Prompt

```
Using the merged MEPS data, compute the weighted uninsurance rate
(INSCOV == 3) by year, overall and broken down by:
  - age group (18-25, 26-34, 35-44, 45-54, 55-64)
  - poverty category
  - race/ethnicity
  - employment status
  - Census region

Use person weights (PERWT) for nationally representative estimates.
Save the aggregated statistics as CSV files.
```

In [ ]:
# ============================================================
# Compute weighted uninsurance rate
# ============================================================

def weighted_uninsurance_rate(df, group_cols=None):
    """
    Compute weighted uninsurance rate.
    
    Uninsurance rate = sum(PERWT where INSCOV==3) / sum(PERWT)
    
    This is the CORRECT formula using MEPS person-level weights.
    Unweighted estimates are BIASED because MEPS oversamples
    certain populations.
    """
    if group_cols is None:
        group_cols = ['YEAR']
    
    df = df.dropna(subset=['INSCOV', 'PERWT'])
    df = df[df['PERWT'] > 0]
    
    grouped = df.groupby(group_cols)
    
    numerator = grouped.apply(
        lambda g: g.loc[g['INSCOV'] == 3, 'PERWT'].sum()
    ).reset_index(name='uninsured_weighted')
    
    denominator = grouped['PERWT'].sum().reset_index(name='total_weighted')
    
    result = numerator.merge(denominator, on=group_cols)
    result['uninsurance_rate'] = result['uninsured_weighted'] / result['total_weighted']
    
    return result

print('Weighted rate function defined.')

In [ ]:
# Overall uninsurance rate by year
overall = weighted_uninsurance_rate(panel, ['YEAR'])
overall['rate_pct'] = overall['uninsurance_rate'] * 100

print('Overall uninsurance rate by year:')
print(overall[['YEAR', 'rate_pct']].to_string(index=False, float_format='%.1f'))

In [ ]:
# Create age group variable
def assign_age_group(age):
    if age < 18: return '<18'
    elif age <= 25: return '18-25'
    elif age <= 34: return '26-34'
    elif age <= 44: return '35-44'
    elif age <= 54: return '45-54'
    elif age <= 64: return '55-64'
    else: return '65+'

panel['AGE_GROUP'] = panel['AGE'].apply(assign_age_group)

# By age group
by_age = weighted_uninsurance_rate(panel, ['YEAR', 'AGE_GROUP'])
by_age['rate_pct'] = by_age['uninsurance_rate'] * 100

print('Uninsurance rate by age group (2013):')
print(by_age[by_age['YEAR'] == 2013][['AGE_GROUP', 'rate_pct']].to_string(index=False, float_format='%.1f'))

In [ ]:
# Poverty category labels
POVCAT_LABELS = {
    1: 'Poor (<100% FPL)',
    2: 'Near Poor (100-125%)',
    3: 'Low Income (125-200%)',
    4: 'Middle Income (200-400%)',
    5: 'High Income (>400%)',
}
panel['POVCAT_LABEL'] = panel['POVCAT'].map(POVCAT_LABELS)

# Race/ethnicity labels
RACE_LABELS = {
    1: 'Hispanic',
    2: 'White Non-Hispanic',
    3: 'Black Non-Hispanic',
    4: 'Asian Non-Hispanic',
    5: 'Other/Multiple',
}
panel['RACE_LABEL'] = panel['RACETHX'].map(RACE_LABELS)

# Region labels
REGION_LABELS = {1: 'Northeast', 2: 'Midwest', 3: 'South', 4: 'West'}
panel['REGION_LABEL'] = panel['REGION'].map(REGION_LABELS)

# Employment labels
EMPST_LABELS = {
    1: 'Employed Full-Time',
    2: 'Employed Part-Time',
    3: 'Not Employed',
    4: 'Employed (Inapplicable)',
}

print('Labels assigned.')

In [ ]:
# Compute all breakdowns
by_poverty = weighted_uninsurance_rate(panel, ['YEAR', 'POVCAT_LABEL'])
by_poverty['rate_pct'] = by_poverty['uninsurance_rate'] * 100

by_race = weighted_uninsurance_rate(panel, ['YEAR', 'RACE_LABEL'])
by_race['rate_pct'] = by_race['uninsurance_rate'] * 100

by_region = weighted_uninsurance_rate(panel, ['YEAR', 'REGION_LABEL'])
by_region['rate_pct'] = by_region['uninsurance_rate'] * 100

print('All breakdowns computed.')
print(f'  Overall: {len(overall)} year-rows')
print(f'  By age: {len(by_age)} group-year rows')
print(f'  By poverty: {len(by_poverty)} group-year rows')
print(f'  By race: {len(by_race)} group-year rows')
print(f'  By region: {len(by_region)} group-year rows')

In [ ]:
# Save all as CSV
overall.to_csv('meps_overall_rate.csv', index=False)
by_age.to_csv('meps_rate_by_age.csv', index=False)
by_poverty.to_csv('meps_rate_by_poverty.csv', index=False)
by_race.to_csv('meps_rate_by_race.csv', index=False)
by_region.to_csv('meps_rate_by_region.csv', index=False)

print('All CSV files saved.')

### Checkpoint (30 min)

You should now have CSV files with group-level weighted uninsurance rates.

**Validation:** Does the 2013 overall rate match approximately 13-14%? Does the 65+ rate show approximately 0%?

## Step 3: Visualization (Prompt 3)

### The Claude Code Prompt

```
Generate 6 publication-quality figures showing uninsurance
rate trends 2000-2023. Use matplotlib with a clean academic style.
Add vertical dashed lines for: ACA passage (2010),
ACA coverage expansion (2014), COVID-19 (2020).
Each figure: clear axis labels, legend, title.
Save as both PNG (300 DPI) and PDF.
```

In [ ]:
# ============================================================
# Helper: add policy event annotations
# ============================================================

def add_policy_annotations(ax, ymin=None, ymax=None):
    """Add ACA and COVID vertical lines to a time series plot."""
    for year, label, color in [
        (2010, 'ACA Passed', 'steelblue'),
        (2014, 'ACA Expansion', 'darkgreen'),
        (2020, 'COVID-19', 'firebrick'),
    ]:
        ax.axvline(x=year, color=color, linestyle='--', alpha=0.5, linewidth=1)
        if ymax is not None:
            ax.text(year + 0.2, ymax * 0.95, label, fontsize=8,
                    color=color, rotation=90, va='top', ha='left', alpha=0.7)

print('Annotation helper defined.')

In [ ]:
# Figure 1: Overall uninsurance rate
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(overall['YEAR'], overall['rate_pct'], 'o-', color='steelblue',
        linewidth=2, markersize=5, label='Overall')

add_policy_annotations(ax, ymax=overall['rate_pct'].max() * 1.05)

ax.set_xlabel('Year')
ax.set_ylabel('Uninsurance Rate (%)')
ax.set_title('US Uninsurance Rate, 2000–2023 (MEPS)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_overall_uninsurance.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_overall_uninsurance.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# Figure 2: By age group
fig, ax = plt.subplots(figsize=(10, 6))

age_order = ['18-25', '26-34', '35-44', '45-54', '55-64']
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(age_order)))

for i, grp in enumerate(age_order):
    subset = by_age[by_age['AGE_GROUP'] == grp].sort_values('YEAR')
    ax.plot(subset['YEAR'], subset['rate_pct'], 'o-',
            color=colors[i], linewidth=1.5, markersize=4, label=grp)

add_policy_annotations(ax, ymax=by_age['rate_pct'].max() * 1.05)

ax.set_xlabel('Year')
ax.set_ylabel('Uninsurance Rate (%)')
ax.set_title('Uninsurance Rate by Age Group, 2000–2023 (MEPS)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(title='Age Group', loc='upper right')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_age.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_age.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# Figure 3: By poverty category
fig, ax = plt.subplots(figsize=(10, 6))

pov_order = ['Poor (<100% FPL)', 'Near Poor (100-125%)', 'Low Income (125-200%)',
             'Middle Income (200-400%)', 'High Income (>400%)']
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(pov_order)))

for i, grp in enumerate(pov_order):
    subset = by_poverty[by_poverty['POVCAT_LABEL'] == grp].sort_values('YEAR')
    if len(subset) > 0:
        ax.plot(subset['YEAR'], subset['rate_pct'], 'o-',
                color=colors[i], linewidth=1.5, markersize=4, label=grp)

add_policy_annotations(ax, ymax=by_poverty['rate_pct'].max() * 1.05)

ax.set_xlabel('Year')
ax.set_ylabel('Uninsurance Rate (%)')
ax.set_title('Uninsurance Rate by Income Level, 2000–2023 (MEPS)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(title='Poverty Category', loc='upper right', fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_poverty.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_poverty.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# Figure 4: By race/ethnicity
fig, ax = plt.subplots(figsize=(10, 6))

race_order = ['Hispanic', 'Black Non-Hispanic', 'White Non-Hispanic', 'Asian Non-Hispanic']
race_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

for i, grp in enumerate(race_order):
    subset = by_race[by_race['RACE_LABEL'] == grp].sort_values('YEAR')
    if len(subset) > 0:
        ax.plot(subset['YEAR'], subset['rate_pct'], 'o-',
                color=race_colors[i], linewidth=1.5, markersize=4, label=grp)

add_policy_annotations(ax, ymax=by_race['rate_pct'].max() * 1.05)

ax.set_xlabel('Year')
ax.set_ylabel('Uninsurance Rate (%)')
ax.set_title('Uninsurance Rate by Race/Ethnicity, 2000–2023 (MEPS)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(title='Race/Ethnicity', loc='upper right')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_race.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_race.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# Figure 5: By region
fig, ax = plt.subplots(figsize=(10, 6))

region_order = ['South', 'West', 'Midwest', 'Northeast']
region_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

for i, grp in enumerate(region_order):
    subset = by_region[by_region['REGION_LABEL'] == grp].sort_values('YEAR')
    if len(subset) > 0:
        ax.plot(subset['YEAR'], subset['rate_pct'], 'o-',
                color=region_colors[i], linewidth=1.5, markersize=4, label=grp)

add_policy_annotations(ax, ymax=by_region['rate_pct'].max() * 1.05)

ax.set_xlabel('Year')
ax.set_ylabel('Uninsurance Rate (%)')
ax.set_title('Uninsurance Rate by Census Region, 2000–2023 (MEPS)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(title='Region', loc='upper right')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_region.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_uninsurance_by_region.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

In [ ]:
# Figure 6: Heatmap (Region x Year)
pivot = by_region.pivot_table(
    index='REGION_LABEL', columns='YEAR', values='rate_pct'
).reindex(['South', 'West', 'Midwest', 'Northeast'])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns.astype(int), rotation=45, fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Uninsurance Rate (%)')

ax.set_title('Uninsurance Rate by Region and Year (MEPS)')

# Add ACA annotation
aca_idx = list(pivot.columns).index(2014) if 2014 in pivot.columns else None
if aca_idx:
    ax.axvline(x=aca_idx - 0.5, color='white', linewidth=2, linestyle='--')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_uninsurance_heatmap.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_uninsurance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

### Checkpoint (45 min)

You should have at least 3 figures generated. Check:
- Do the trends tell the right story? (Decline after 2014, stable late 2010s)
- Are the ACA/COVID annotation lines in the right places?
- Is the legend readable?

## Step 4: Validation

Cross-check your MEPS estimates against published benchmarks.

In [ ]:
# ============================================================
# Validation: Compare to Census Bureau CPS ASEC
# ============================================================

# Known uninsurance rates from Census Bureau
census_benchmarks = {
    2010: 16.3,
    2013: 13.4,
    2016: 8.8,
    2019: 8.0,
    2022: 7.9,
}

print('Validation: MEPS vs. Census Bureau CPS ASEC')
print(f'{"Year":>6} {"Census (%)":>12} {"MEPS (%)":>12} {"Diff (pp)":>12}')
print('-' * 44)

for year, census_rate in sorted(census_benchmarks.items()):
    meps_row = overall[overall['YEAR'] == year]
    if len(meps_row) > 0:
        meps_rate = meps_row['rate_pct'].values[0]
        diff = meps_rate - census_rate
        print(f'{year:>6} {census_rate:>11.1f}% {meps_rate:>11.1f}% {diff:>+11.1f}')
    else:
        print(f'{year:>6} {census_rate:>11.1f}%   (not available)')

print('\nNote: MEPS and CPS use different survey designs.')
print('Differences of +/- 2 percentage points are expected.')
print('The KEY check: both should show the same TREND (sharp decline after 2014).')

## Discussion

### Key Findings

1. **The ACA worked:** Uninsurance rates dropped dramatically after the 2014 coverage expansion
2. **Unequal gains:** Low-income and minority groups saw the largest improvements
3. **Persistent gaps:** Even after the ACA, significant disparities remain by race, income, and region
4. **The South stands out:** Southern states, which were less likely to expand Medicaid, have consistently higher rates

### What Claude Code Did vs. What Stayed Human

| Claude Code | Economist |
|---|---|
| Downloaded 24 files from AHRQ | Posed the research question |
| Built the variable crosswalk | Specified which breakdowns to compute |
| Computed weighted statistics | Validated against Census Bureau |
| Generated 6 figures | Interpreted results against economic theory |
| Handled format obstacles | Made the judgment calls on figure design |

### Connection to Course Themes

- **Heterogeneous-agent models (Lec06):** These figures provide the empirical targets for calibrating HA models of health insurance markets
- **Pillar 5 (Lec09):** We validated every step against known benchmarks — the verification pillar is the economist's contribution
- **PaulGP arc:** Problem → Infrastructure → Architecture → Capability → Insights — the same arc that works for mortgage data works for health insurance data

### Checkpoint (60 min)

**You should now have:**
- A merged MEPS panel (parquet file)
- CSV files with group-level statistics
- 6 publication-quality figures
- Validation against Census Bureau benchmarks
- Confidence that the trends are correct

**Congratulations!** You have completed a full data analysis workflow using Claude Code as your research assistant.